### 0. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

os.chdir("../")
from scripts import utils
from pathlib import Path
import matplotlib.gridspec as gridspec
from tqdm.auto import tqdm

In [2]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
from mlxtend.evaluate import feature_importance_permutation
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs
from sklearn.utils.estimator_checks import check_estimator
from mlxtend.feature_selection import (
    SequentialFeatureSelector,
)
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
import matplotlib.ticker as ticker
import distclassipy as dcpy
from distclassipy.anomaly import DistanceAnomaly

In [4]:
from sklearn.base import BaseEstimator, OutlierMixin

In [5]:
epsilon = np.finfo(np.float32).eps

In [6]:
with open("settings.txt") as f:
    settings_dict = json.load(f)
seed_val = settings_dict["seed_choice"]
np.random.seed(seed_val)
sns_dict = settings_dict["sns_dict"]
sns.set_theme(**sns_dict)

In [7]:
unique_metrics = ['euclidean',
 'braycurtis',
 'canberra',
 'cityblock',
 'chebyshev',
 'clark',
 'correlation',
 'cosine',
 'hellinger',
 'jaccard',
 'lorentzian',
 # 'marylandbridge',
 'meehl',
 'motyka',
 'soergel',
 'wave_hedges',
 'kulczynski',
 # 'add_chisq'
                 ]


final_features = [
    "SPM_A_Y",
    "Multiband_period",
    "r-i",
    "Harmonics_phase_4_i",
    "Harmonics_phase_2_r",
    "Power_rate_4",
]

final_features = [
    # "g-r",
    # "r-i",
    # "i-z",

# 'SPM_A_g', 'SPM_gamma_g', 'SPM_beta_g',
#        'SPM_tau_rise_g', 'SPM_tau_fall_g', 'SPM_A_r', 'SPM_gamma_r',
#     'SPM_beta_r', 'SPM_tau_rise_r', 'SPM_tau_fall_r', 

# 'SPM_A_g', 'SPM_gamma_g', 'SPM_beta_g',
#        'SPM_tau_rise_g', 'SPM_tau_fall_g', 'SPM_A_r', 'SPM_gamma_r',
#        'SPM_beta_r', 'SPM_tau_rise_r', 'SPM_tau_fall_r', 'SPM_A_i',
#        'SPM_gamma_i', 'SPM_beta_i', 'SPM_tau_rise_i', 'SPM_tau_fall_i',
#        'SPM_A_z', 'SPM_gamma_z', 'SPM_beta_z', 'SPM_tau_rise_z',
#        'SPM_tau_fall_z', 'SPM_A_Y', 'SPM_gamma_Y', 'SPM_beta_Y',
#        'SPM_tau_rise_Y', 'SPM_tau_fall_Y', 'SPM_chi_r', 'SPM_chi_i',
#        'SPM_chi_z',
]

### 1. Data Prep
- Make a knowns/inlier dataset with 4 classes (CEP, RR, DSCT and EB): ```X_knowns_df``` and ```y_knowns_df```
- Make an unknowns/outlier/anomaly dataset with other classes: ```X_anom_df``` and ```y_anom_df```

An ```X_all_df``` and ```y_all_df``` contains both of these.

In [8]:
knowns = pd.read_parquet("data/reduced_balancedfeatures_LATEST.parquet")
knowns = knowns.sample(frac=1) #shuffle

y_knowns_df = knowns["class"]
X_knowns_df = knowns.loc[:, final_features]


unknowns = pd.read_parquet("data/otherclassobjs_features.parquet")
unknowns.index.name = "snid"
unknowns_lc_df = pd.read_parquet("data/otherclassobjs.parquet")
unknowns_lc_df.index.name = "snid"

unknowns_lc_df=unknowns_lc_df[~unknowns_lc_df["class"].isin(['d-Sct', 'Cepheid', 'EB', 'RRL'])]
unknowns=unknowns.loc[unknowns_lc_df.index]


unknowns_lc_df = unknowns_lc_df.loc[unknowns.index]

assert (unknowns.index == unknowns_lc_df.index).all()


X_anom_df = unknowns.loc[:, X_knowns_df.columns].dropna()
X_anom_df = X_anom_df.drop(np.intersect1d(X_anom_df.index, X_knowns_df.index))
y_anom_df = unknowns_lc_df.loc[X_anom_df.index]["class"]
X_anom_df=X_anom_df.loc[y_anom_df.index]

In [9]:
X_all_df = pd.concat([X_knowns_df, X_anom_df]).sample(frac=1)
y_all_df = pd.concat([y_knowns_df, y_anom_df]).loc[X_all_df.index]

---
---

In [10]:
# new_unknowns = ["KN_B19", "KN_K17"]

# new_knowns = ["SNIb+HostXT_V19", 
# "SLSN-I_no_host", 
# "SNIa-SALT3", 
# "SNIc+HostXT_V19", 
# "SNIc-Templates", 
# "SNIIn+HostXT_V19", 
# "SNIax", 
# "SLSN-I+host"]

new_unknowns = ["PISN-STELLA_HYDROGENIC",
"SLSN-I_no_host",
"SLSN-I+host",
"ILOT",
"CART",
"uLens-Single_PyLIMA",
"uLens-Single-GenLens",
"KN_B19",
"KN_K17"]


new_knowns = ["SNIc-Templates",
"SNIb-Templates",
"SNIax",
"SNIc+HostXT_V19",
"SNII-NMF",
"SNIIn+HostXT_V19",
"SNIcBL+HostXT_V19",
"PISN-MOSFIT",
"PISN-STELLA_HECORE",
"SNIb+HostXT_V19",
"SNIa-91bg",
"uLens-Binary",
"SNIIb+HostXT_V19",
"SNIIn-MOSFIT",
"SNII+HostXT_V19",
"TDE",
"SNIa-SALT3",
"Mdwarf-flare",
"dwarf-nova",
"SL-SN1a",
"SL-SNII",
"SL-SNIc",
"SL-SNIb"]




In [11]:
new_unknown_idx = y_all_df.isin(new_unknowns)
new_known_idx = y_all_df.isin(new_knowns)

In [12]:
del(X_knowns_df, X_anom_df)

In [13]:
X_knowns_df = X_all_df.loc[new_known_idx]
y_knowns_df = y_all_df.loc[new_known_idx]
X_anom_df = X_all_df.loc[new_unknown_idx]
y_anom_df = y_all_df.loc[new_unknown_idx]

In [14]:
del(X_all_df)

In [15]:
X_all_df = pd.concat([X_knowns_df, X_anom_df]).sample(frac=1)
y_all_df = pd.concat([y_knowns_df, y_anom_df]).loc[X_all_df.index]

---
---

In [16]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.metrics import mean_squared_error

# Helper class to make the Autoencoder scikit-learn compatible
class AutoencoderAnomalyDetector(BaseEstimator, OutlierMixin):
    def __init__(self, encoding_dim=3, epochs=50, batch_size=32, verbose=0):
        self.encoding_dim = encoding_dim
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose

    def fit(self, X, y=None):
        input_dim = X.shape[1]
        
        # Encoder
        input_layer = Input(shape=(input_dim,))
        encoder = Dense(int(input_dim / 2), activation="relu")(input_layer)
        encoder = Dense(self.encoding_dim, activation="relu")(encoder)
        
        # Decoder
        decoder = Dense(int(input_dim / 2), activation="relu")(encoder)
        decoder = Dense(input_dim, activation="sigmoid")(decoder)
        
        self.model_ = Model(inputs=input_layer, outputs=decoder)
        self.model_.compile(optimizer='adam', loss='mean_squared_error')
        
        # Train on the normal data
        self.model_.fit(X, X,
                        epochs=self.epochs,
                        batch_size=self.batch_size,
                        shuffle=True,
                        verbose=self.verbose)
        return self

    def decision_function(self, X):
        # Higher reconstruction error -> more anomalous
        reconstructions = self.model_.predict(X, verbose=self.verbose)
        mse = np.mean(np.power(X - reconstructions, 2), axis=1)
        return mse


In [17]:
# Create a dictionary of all models to test
models_to_test = {
    "DistClassiPy (min-med)": DistanceAnomaly(
        metrics=unique_metrics,
        cluster_agg='min',
        metric_agg='median',
        normalize_scores=True
    ),
    "IsolationForest": IsolationForest(
        contamination='auto', 
        random_state=seed_val
    ),
    "LocalOutlierFactor": LocalOutlierFactor(
        n_neighbors=20, 
        metric='jaccard',
        novelty=True,
        contamination='auto'
    ),
    "Autoencoder": AutoencoderAnomalyDetector(
        encoding_dim=3, # Can be tuned
        epochs=50,
        verbose=0
    ),
    "Euclidean baseline (mean)": DistanceAnomaly(
        metrics=["Euclidean"],
        cluster_agg='mean',
        metric_agg='mean',
        normalize_scores=True
    ),

}

In [18]:
results_df = pd.DataFrame(index=X_all_df.index)
results_df['class'] = y_all_df
results_df['status'] = results_df['class'].apply(
    lambda x: 'normal' if x in sorted(y_knowns_df.unique()) else 'anomalous'
)

# Loop through models, fit, score, and store results
for model_name, model in tqdm(models_to_test.items(), desc="Running Models"):
    print(f"--- Training {model_name} ---")
    
    # Fit the model on the NORMAL data only
    model.fit(X_knowns_df.to_numpy(), y_knowns_df.to_numpy())
    
    # Get anomaly scores for ALL data
    # Note: IF and LOF score lower for anomalies, so we negate them.
    # Your method and AE score higher for anomalies, so no change is needed.
    scores = model.decision_function(X_all_df.to_numpy())
    
    if model_name in ["IsolationForest", "LocalOutlierFactor"]:
        scores = -scores # Lower is more anomalous for these, so we flip
        
    results_df[f'score_{model_name}'] = scores

Running Models:   0%|          | 0/5 [00:00<?, ?it/s]

--- Training DistClassiPy (min-med) ---


ValueError: Found array with 0 feature(s) (shape=(2029, 0)) while a minimum of 1 is required by DistanceMetricClassifier.

In [ ]:
results_df.sort_values(by=["score_Euclidean baseline (mean)"],ascending=False)["status"].iloc[:100].value_counts()

In [ ]:
results_df.sort_values(by=["score_DistClassiPy (min-med)"],ascending=False)["status"].iloc[:100].value_counts()

In [ ]:
results_df.sort_values(by=["score_IsolationForest"],ascending=False)["status"].iloc[:100].value_counts()

In [ ]:
results_df.sort_values(by=["score_LocalOutlierFactor"],ascending=False)["status"].iloc[:100].value_counts()

In [ ]:
results_df.sort_values(by=["score_Autoencoder"],ascending=False)["status"].iloc[:100].value_counts()

In [ ]:
results_df["status"].value_counts()

# normal       2732
# anomalous    1586

In [ ]:
# # Example: Check top 200 candidates for each model
# top_n = 200
# for model_name in models_to_test.keys():
#     score_col = f'score_{model_name}'
#     print(f"\n--- Top {top_n} candidates for {model_name} ---")
    
#     top_candidates = results_df.sort_values(score_col, ascending=False).head(top_n)
#     print(top_candidates['status'].value_counts(normalize=True))

In [ ]:
dcpypercent = []
ifpercent = []
lofpercent = []
aepercent = []
methods_percent = {}

# baseline = []

for model_name in tqdm(models_to_test.keys(),desc="method"):
    methods_percent[model_name] = []
    score_col = f'score_{model_name}'
    # normalize=True
    for top_n in tqdm(range(1, len(results_df)+1), desc="topn",leave=False):
        candidates = results_df.sort_values(score_col, ascending=False)
        top_candidates = candidates.head(top_n)
        valcounts = top_candidates['status'].value_counts(normalize=False,ascending=True)
        tot_anom = candidates["status"].value_counts()["anomalous"]
        if "anomalous" in valcounts.index:
            frac = valcounts["anomalous"] / tot_anom
        else:
            frac = 0
        methods_percent[model_name].append(frac)

In [ ]:
endnum = 500
# endum = len(results_df)+1

top_ns = np.arange(start=1, stop=endnum+1)


for model_name in models_to_test.keys():
    plt.plot(top_ns, methods_percent[model_name][:endnum], label=model_name)
plt.ylabel("Percentage of anomalies correctly identified")
plt.xlabel("Depth of search")
plt.legend()
plt.show()

In [ ]:
results_df.sort_values(by="score_DistClassiPy (min-med)",ascending=False)